In [ ]:
import nltk
nltk.download('conll2002')
from nltk.corpus import conll2002

In [ ]:
esp_train = conll2002.iob_sents('esp.train') # Train, ned.train => Neerlandès
esp_val = conll2002.iob_sents('esp.testa') # Dev
esp_test = conll2002.iob_sents('esp.testb') # Test

In [7]:
ned_train = conll2002.iob_sents('ned.train') # Train, ned.train => Neerlandès
ned_val = conll2002.iob_sents('ned.testa') # Dev
ned_test = conll2002.iob_sents('ned.testb')

In [12]:
def prepare_data(sents):
    return [[(word,bio) for word, pos, bio in sent] for sent in sents]

In [14]:
esp_train_net = prepare_data(esp_train)
esp_val_net = prepare_data(esp_val)
esp_test_net = prepare_data(esp_test)
ned_train_net = prepare_data(ned_train)
ned_val_net = prepare_data(ned_val)
ned_test_net = prepare_data(ned_train)

In [16]:
def bio_to_io(sentence):
    new_sentence = []
    
    for word, tag in sentence:
        if tag.startswith("B-") or tag.startswith("I-"):
            new_tag = "I-" + tag[2:]
        else:
            new_tag = tag  # "O"
            
        new_sentence.append((word, new_tag))
    
    return new_sentence

In [17]:
def bio_to_biow(sentence):
    new_sentence = []
    i = 0
    n = len(sentence)
    
    while i < n:
        word, tag = sentence[i]
        
        if tag.startswith("B-"):
            entity_type = tag[2:]
            j = i + 1
            
            # trobar final de l'entitat
            while j < n and sentence[j][1] == f"I-{entity_type}":
                j += 1
            
            length = j - i
            
            if length == 1:
                new_sentence.append((word, f"W-{entity_type}"))
            else:
                new_sentence.append((word, f"B-{entity_type}"))
                for k in range(i+1, j):
                    new_sentence.append((sentence[k][0], f"I-{entity_type}"))
            
            i = j
        
        else:
            new_sentence.append((word, tag))
            i += 1
    
    return new_sentence

In [18]:
def bio_to_bioew(sentence):
    new_sentence = []
    i = 0
    n = len(sentence)
    
    while i < n:
        word, tag = sentence[i]
        
        if tag.startswith("B-"):
            entity_type = tag[2:]
            j = i + 1
            
            # trobar final de l'entitat
            while j < n and sentence[j][1] == f"I-{entity_type}":
                j += 1
            
            length = j - i
            
            if length == 1:
                new_sentence.append((word, f"W-{entity_type}"))
            else:
                # primer
                new_sentence.append((word, f"B-{entity_type}"))
                
                # mig
                for k in range(i+1, j-1):
                    new_sentence.append((sentence[k][0], f"I-{entity_type}"))
                
                # últim
                last_word = sentence[j-1][0]
                new_sentence.append((last_word, f"E-{entity_type}"))
            
            i = j
        
        else:
            new_sentence.append((word, tag))
            i += 1
    
    return new_sentence